# Data Cleaning & Transformation (Silver Layer)

In [0]:
# Import libraries
from pyspark.sql import functions as F
from pyspark.sql.window import Window 

In [0]:
# Reading the bronze table
bronze_df = spark.table("gsma_workspace.global_stocks_schema.bronze_global_stock_data")

# Verify
bronze_df.show(5)

## Data Cleaning

In [0]:
# Handle missing or invalid values
# Check for duplicate rows
total_count = bronze_df.count()
distinct_count = bronze_df.dropDuplicates().count()

print(f"Total: {total_count}, Distinct: {distinct_count}")

In [0]:
# Check for invalid price values
bronze_df.filter(
    (F.col("Close") <= 0) |
    (F.col("Open") <= 0) |
    (F.col("High") <= 0) |
    (F.col("Low") <= 0)
).show()

In [0]:
# Check logical inconsistencies
# Stock prices must follow:
# High ≥ Open, Close, Low
# Low ≤ Open, Close, High

bronze_df.filter(
    (F.col("High") < F.col("Low")) |
    (F.col("High") < F.col("Open")) |
    (F.col("High") < F.col("Close")) |
    (F.col("Low") > F.col("Open")) |
    (F.col("Low") > F.col("Close"))
).show()


In [0]:
# Check for invalid dates
bronze_df.filter(col("Date").isNull()).show()

## Data Transformation

In [0]:
# Add Country column based on the Index
bronze_df = bronze_df.withColumn("Country", 
                                 F.when(F.col("Index") == "SP500", "USA")
                                 .when(F.col("Index") == "NIFTY50", "India")
                                 .when(F.col("Index") == "BIST100", "TURKEY")
                                 .when(F.col("Index") == "BOVESPA", "Brazil")
                                 .when(F.col("Index") == "DAX40", "Germany")
                                 .when(F.col("Index") == "FTSE100", "UK")
                                 .when(F.col("Index") == "IDX", "Indonesia")
                                 .when(F.col("Index") == "NIKKEI225", "Japan")
                                 .when(F.col("Index") == "SSE", "China")
                                 .when(F.col("Index") == "TADAWUL", "SaudiArabia")
                                 .otherwise("Unknown"))

# Add Year column
bronze_df = bronze_df.withColumn("Year", F.year(F.col("Date")))

# Add Month column
bronze_df = bronze_df.withColumn("Month", F.month(F.col("Date")))

# Calculate and add Daily_Return column
bronze_df = bronze_df.withColumn("Daily_Return", F.round(F.col("Close") / F.lag(F.col("Close")).over(Window.partitionBy("Index").orderBy("Date")) - 1, 4))

In [0]:
# View changes
display(bronze_df)

In [0]:
# Find Rows with Nulls in any column
bronze_df.filter(" OR ".join([f"`{c}` IS NULL" for c in bronze_df.columns])).show()

In [0]:
# Daily_Return column has null values due to first day of the month
# Fill the null with 0
bronze_df = bronze_df.fillna({"Daily_Return": 0})

# Check again
bronze_df.filter(" OR ".join([f"`{c}` IS NULL" for c in bronze_df.columns])).show()

In [0]:
# Save the dataframe to a new Silver Delta Table
bronze_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gsma_workspace.global_stocks_schema.silver_global_stock_data")

In [0]:
%sql
-- Validate table created in SQL
SELECT COUNT(*) FROM gsma_workspace.global_stocks_schema.silver_global_stock_data;